In [7]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
from pandas.tseries.offsets import MonthBegin
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error
import optuna
from optuna.samplers import TPESampler
from sklearn.impute import SimpleImputer
import warnings

warnings.filterwarnings("ignore")

# ────────── RUTAS Y CONFIG ──────────
BASE      = Path(r"C:/Users/const/OneDrive/Desktop/labo 3")
VENTAS    = BASE / "sell-in.txt"
LISTA     = BASE / "780_a_predecir.txt"
OUT_SUB   = BASE / "submission_mega_ensemble.csv"
SEED      = 14
MAX_LAG   = 12
ROLL      = [3, 6, 12]
VAL_MONTHS= 12
N_TRIALS  = 25
# ────────────────────────────────────

random.seed(SEED)
np.random.seed(SEED)

# 1) Leer y preparar datos
df = pd.read_csv(VENTAS, sep="\t", engine="python")
df.columns = df.columns.str.strip()
df = df.rename(columns={next(c for c in df.columns if "period" in c.lower()): "periodo"})
df["periodo"] = df["periodo"].astype(str).str.zfill(6)
df["fecha"]   = pd.to_datetime(df["periodo"].str[:4] + df["periodo"].str[4:6] + "01", 
                                format="%Y%m%d", errors="coerce")

with open(LISTA) as f:
    products = [int(l) for l in f if l.strip().isdigit()]

df = df[df["product_id"].isin(products) & df["fecha"].notna()]

# 2) Serie mensual tn por SKU
mensual = (
    df.groupby(["product_id","fecha"])["tn"]
      .sum()
      .unstack(0)
      .asfreq("MS", fill_value=0)
)

# 3) Crear features
records = []
for pid in mensual.columns:
    s = mensual[pid]
    for i in range(MAX_LAG, len(s)):
        dt = s.index[i]
        r = {"product_id": pid, "fecha": dt, "target": s.iloc[i]}
        # lags y diffs
        for lag in range(1, MAX_LAG+1):
            r[f"lag_{lag}"] = s.iloc[i-lag]
        for lag in [1,3,6]:
            r[f"diff_{lag}"] = s.iloc[i-lag] - s.iloc[i-lag-1]
        # rolling stats y ratios
        for w in ROLL:
            win = s.iloc[i-w:i]
            r[f"mean_{w}"] = win.mean()
            r[f"std_{w}"]  = win.std()
            r[f"ratio_{w}"] = win.mean() / (s.iloc[i-1] + 1e-6)
        # calendar
        r["month"] = dt.month
        r["year"]  = dt.year
        records.append(r)

df_feat = pd.DataFrame(records).fillna(0)

# 4) Split temporal
cut = mensual.index.max() - pd.DateOffset(months=VAL_MONTHS)
train = df_feat[df_feat["fecha"] <= cut]
valid = df_feat[df_feat["fecha"] >  cut]

X_tr = train.drop(["target","fecha"], axis=1)
y_tr = train["target"]
X_va = valid.drop(["target","fecha"], axis=1)
y_va = valid["target"]

FEATURES = X_tr.columns.tolist()
CAT_FEAT = ["product_id","month","year"]

# 5) Preprocesamiento
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore"), CAT_FEAT),
], remainder="passthrough")

# 6) Tuning de LGBM y XGB con Optuna
def ensemble_objective(trial):
    lgb_params = {
        "learning_rate": trial.suggest_loguniform("lgb_lr", 1e-3, 1e-1),
        "num_leaves":    trial.suggest_int("lgb_leaves", 16, 64),
        "random_state":  SEED,
        "n_jobs":        -1
    }
    xgb_params = {
        "learning_rate": trial.suggest_loguniform("xgb_lr", 1e-3, 1e-1),
        "max_depth":     trial.suggest_int("xgb_depth", 3, 8),
        "subsample":     trial.suggest_float("xgb_sub", 0.5, 1.0),
        "random_state":  SEED,
        "n_jobs":        -1
    }
    rf = RandomForestRegressor(random_state=SEED, n_jobs=-1)
    lgbm_pipe = make_pipeline(preprocessor, lgb.LGBMRegressor(**lgb_params))
    xgb_pipe = make_pipeline(preprocessor, XGBRegressor(**xgb_params))
    stack = StackingRegressor(
        estimators=[("rf", rf), ("lgbm", lgbm_pipe), ("xgb", xgb_pipe)],
        final_estimator=ElasticNetCV(cv=3, random_state=SEED),
        cv=3,
        n_jobs=-1,
        passthrough=True
    )
    stack.fit(X_tr, y_tr)
    pred = stack.predict(X_va)
    return mean_absolute_error(y_va, pred)

study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED))
study.optimize(ensemble_objective, n_trials=N_TRIALS)
best = study.best_params

# 7) Modelos base finales
lgbm_final = make_pipeline(
    preprocessor,
    lgb.LGBMRegressor(
        learning_rate=best["lgb_lr"],
        num_leaves=best["lgb_leaves"],
        random_state=SEED, n_jobs=-1
    )
)
xgb_final = make_pipeline(
    preprocessor,
    XGBRegressor(
        learning_rate=best["xgb_lr"],
        max_depth=best["xgb_depth"],
        subsample=best["xgb_sub"],
        random_state=SEED, n_jobs=-1
    )
)
rf_final = make_pipeline(preprocessor, RandomForestRegressor(random_state=SEED, n_jobs=-1))

# 8) Stacking definitivo
stack_final = StackingRegressor(
    estimators=[("rf", rf_final), ("lgbm", lgbm_final), ("xgb", xgb_final)],
    final_estimator=ElasticNetCV(cv=5, random_state=SEED),
    cv=5,
    n_jobs=-1,
    passthrough=True
)
stack_final.fit(X_tr, y_tr)

# 9) Validación
va_pred = stack_final.predict(X_va)
print(f">>> MEGA-stacking MAE en validación: {mean_absolute_error(y_va, va_pred):.4f}")

# 10) Reentrenar con todo
X_all = pd.concat([X_tr, X_va])
y_all = pd.concat([y_tr, y_va])
stack_final.fit(X_all, y_all)

# 11) Predecir siguiente mes
next_dt = mensual.index.max() + MonthBegin()
rows = []
for pid in mensual.columns:
    s = mensual[pid]
    r = {"product_id": pid, "month": next_dt.month, "year": next_dt.year}
    for lag in range(1, MAX_LAG+1):
        r[f"lag_{lag}"] = s.iloc[-lag]
    for lag in [1,3,6]:
        r[f"diff_{lag}"] = s.iloc[-lag] - s.iloc[-lag-1]
    for w in ROLL:
        win = s.iloc[-w:]
        r[f"mean_{w}"]  = win.mean()
        r[f"std_{w}"]   = win.std()
        r[f"ratio_{w}"] = win.mean() / (s.iloc[-1] + 1e-6)
    rows.append(r)

X_pred = pd.DataFrame(rows)[FEATURES]

# 👉 imputamos cualquier NaN antes de predecir
imputer = SimpleImputer(strategy='mean')
X_pred = pd.DataFrame(imputer.fit_transform(X_pred), columns=X_pred.columns)

y_pred = stack_final.predict(X_pred)
y_pred = np.maximum(0, np.round(y_pred,5))

# 12) Guardar submission
pd.DataFrame({
    "product_id": mensual.columns,
    "tn": y_pred
}).to_csv(OUT_SUB, index=False, float_format="%.5f")

print(f"✅ MEGA-ensemble guardado → {OUT_SUB.name}")

[I 2025-07-19 13:26:56,532] A new study created in memory with name: no-name-a4b47175-1153-48df-a38f-72d33db6ce3b
[I 2025-07-19 13:28:00,419] Trial 0 finished with value: 9.464268735183284 and parameters: {'lgb_lr': 0.010663178702639572, 'lgb_leaves': 53, 'xgb_lr': 0.05506242972169045, 'xgb_depth': 3, 'xgb_sub': 0.6548679627526021}. Best is trial 0 with value: 9.464268735183284.
[I 2025-07-19 13:28:57,094] Trial 1 finished with value: 9.464268240779996 and parameters: {'lgb_lr': 0.0822635559887955, 'lgb_leaves': 41, 'xgb_lr': 0.004330807196196427, 'xgb_depth': 6, 'xgb_sub': 0.61062747121388}. Best is trial 1 with value: 9.464268240779996.
[I 2025-07-19 13:29:51,607] Trial 2 finished with value: 9.464270545675959 and parameters: {'lgb_lr': 0.041016888850412554, 'lgb_leaves': 32, 'xgb_lr': 0.011961281137256929, 'xgb_depth': 3, 'xgb_sub': 0.8365762390902627}. Best is trial 1 with value: 9.464268240779996.
[I 2025-07-19 13:30:48,406] Trial 3 finished with value: 9.464268397331598 and param

>>> MEGA-stacking MAE en validación: 9.4642
✅ MEGA-ensemble guardado → submission_mega_ensemble.csv
